# Data Sources

## 1) GRID3 – Nigeria Operational State Boundaries (Adm1)
State boundary polygons (36 states + FCT) were obtained from the GRID3 Data Hub. The dataset provides standardized administrative level 1 geometries and attributes suitable for spatial analysis and mapping. It is released under a CC BY 4.0 license.
Link: https://data.grid3.org/datasets/GRID3::grid3-nga-operational-state-boundaries-/about

## 2) Global Data Lab – Yearly Average Surface Temperature (°C)
Annual average surface temperature data for Nigeria were sourced from the Global Data Lab Geospatial Data Portal (v0.2). The dataset provides yearly temperature estimates at national and subnational levels, downloadable in tabular format for integration with boundary data.
Link: https://globaldatalab.org/geos/download/surfacetempyear/NGA/?levels=1+4

## 3) Data Preparation and Integration

In [3]:
import geopandas as gpd
import pandas as pd
import re


In [14]:
# Load Datasets
ng = gpd.read_file(r"alt_data/NGA_State_Boundaries_V2_-6583428934468491568/grid3_nga_boundary_vaccstates.shp")
av_temp = pd.read_csv(r"alt_data/GDL-Yearly-Average-Surface-Temperature-(ºC) nigeria_state.csv")

In [16]:
ng.head()

,globalid,uniq_id,timestamp,editor,statename,statecode,capcity,source,geozone,geometry
0,c46ae452-d6b6-4618-b3f7-ed6015e978d7,1187,2018-12-13,abraham.oluseye,Cross River,CR,Calabar,eHA_Polio,SSZ,"POLYGON ((8.46076 4.75455, 8.45885 4.75393, 8...."
1,0e73256c-2793-44cc-ab1b-7289f145b866,1175,2018-12-13,abraham.oluseye,Abuja FCT,FC,Abuja,eHA_Polio,NCZ,"POLYGON ((7.47762 8.63133, 7.44303 8.59328, 7...."
2,7f0787e4-4518-4486-8bc5-51c5232df9d1,1190,2019-07-29,nuraddeen.isah,Ogun,OG,Abeokuta,eHA_Polio,SWZ,"POLYGON ((4.59755 6.36057, 4.59757 6.35688, 4...."
3,0edbae77-0091-40d9-afdf-7e249ed797fb,1191,2019-07-29,nuraddeen.isah,Oyo,OY,Ibandan,eHA_Polio,SWZ,"POLYGON ((3.99896 7.12657, 3.99755 7.12617, 3...."
4,143c5cd1-9f44-4935-ab8a-891e4c05451e,1177,2018-12-13,abraham.oluseye,Sokoto,SO,Sokoto,eHA_Polio,NWZ,"POLYGON ((4.63265 11.67143, 4.62886 11.66810, ..."


In [32]:
av_temp.head()

,Country,Continent,ISO_Code,Level,GDLCODE,statename,1990,1991,1992,1993,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,Nigeria,Africa,NGA,Subnat,NGAr122,Abia,25.79,25.67,25.69,25.85,...,26.26,26.38,26.52,26.85,26.84,26.62,26.72,27.09,26.98,26.81
1,Nigeria,Africa,NGA,Subnat,NGAr137,Abuja FCT,27.36,27.08,26.67,27.15,...,27.56,27.44,27.61,27.68,28.17,27.50,27.64,27.71,28.08,27.63
2,Nigeria,Africa,NGA,Subnat,NGAr108,Adamawa,27.40,27.16,26.44,27.03,...,27.43,27.18,27.55,27.55,27.50,27.56,27.61,27.67,27.75,26.98
3,Nigeria,Africa,NGA,Subnat,NGAr101,Akwa Ibom,25.64,25.60,25.61,25.78,...,26.13,26.24,26.29,26.70,26.59,26.45,26.61,26.88,26.76,26.59
4,Nigeria,Africa,NGA,Subnat,NGAr102,Anambra,26.55,26.37,26.29,26.50,...,26.90,26.98,27.15,27.47,27.51,27.22,27.36,27.65,27.76,27.46


In [18]:
# Check for name mismatches before joining
ng_names = set(ng["statename"].dropna().unique())
temp_names = set(av_temp["statename"].dropna().unique())

print("In temp but not in boundaries:", sorted(temp_names - ng_names))
print("In boundaries but not in temp:", sorted(ng_names - temp_names))

In temp but not in boundaries: []
In boundaries but not in temp: []


In [19]:
# # Join (left join keeps all states in ng)
gdf = ng.merge(av_temp, on="statename", how="left")

In [22]:
gdf.head()

,globalid,uniq_id,timestamp,editor,statename,statecode,capcity,source,geozone,geometry,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,c46ae452-d6b6-4618-b3f7-ed6015e978d7,1187,2018-12-13,abraham.oluseye,Cross River,CR,Calabar,eHA_Polio,SSZ,"POLYGON ((8.46076 4.75455, 8.45885 4.75393, 8....",...,26.24,26.33,26.50,26.75,26.78,26.57,26.61,26.96,26.91,26.67
1,0e73256c-2793-44cc-ab1b-7289f145b866,1175,2018-12-13,abraham.oluseye,Abuja FCT,FC,Abuja,eHA_Polio,NCZ,"POLYGON ((7.47762 8.63133, 7.44303 8.59328, 7....",...,27.56,27.44,27.61,27.68,28.17,27.50,27.64,27.71,28.08,27.63
2,7f0787e4-4518-4486-8bc5-51c5232df9d1,1190,2019-07-29,nuraddeen.isah,Ogun,OG,Abeokuta,eHA_Polio,SWZ,"POLYGON ((4.59755 6.36057, 4.59757 6.35688, 4....",...,26.69,26.62,26.77,27.21,27.01,26.87,27.11,27.20,27.36,26.98
3,0edbae77-0091-40d9-afdf-7e249ed797fb,1191,2019-07-29,nuraddeen.isah,Oyo,OY,Ibandan,eHA_Polio,SWZ,"POLYGON ((3.99896 7.12657, 3.99755 7.12617, 3....",...,26.66,26.59,26.77,27.08,27.00,26.66,26.86,26.96,27.36,26.85
4,143c5cd1-9f44-4935-ab8a-891e4c05451e,1177,2018-12-13,abraham.oluseye,Sokoto,SO,Sokoto,eHA_Polio,NWZ,"POLYGON ((4.63265 11.67143, 4.62886 11.66810, ...",...,29.30,29.07,28.95,29.21,29.04,28.87,29.02,28.97,29.38,28.58


In [29]:
# Drop Columns 
cols_to_drop = [
    "globalid",
    "timestamp",
    "capacity",   # make sure spelling matches exactly
    "source",
    "editor",
    "Country",
    "ISO_Code",
    "Level",
    "GDLCODE",
    "Continent",
    "capcity",
    "statecode"
]

gdf = gdf.drop(columns=cols_to_drop, errors="ignore")

In [30]:
gdf.head()

,uniq_id,statename,geozone,geometry,1990,1991,1992,1993,1994,1995,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
0,1187,Cross River,SSZ,"POLYGON ((8.46076 4.75455, 8.45885 4.75393, 8....",25.87,25.64,25.65,25.88,25.90,26.00,...,26.24,26.33,26.50,26.75,26.78,26.57,26.61,26.96,26.91,26.67
1,1175,Abuja FCT,NCZ,"POLYGON ((7.47762 8.63133, 7.44303 8.59328, 7....",27.36,27.08,26.67,27.15,27.04,27.14,...,27.56,27.44,27.61,27.68,28.17,27.50,27.64,27.71,28.08,27.63
2,1190,Ogun,SWZ,"POLYGON ((4.59755 6.36057, 4.59757 6.35688, 4....",26.26,26.05,26.01,26.22,26.17,26.37,...,26.69,26.62,26.77,27.21,27.01,26.87,27.11,27.20,27.36,26.98
3,1191,Oyo,SWZ,"POLYGON ((3.99896 7.12657, 3.99755 7.12617, 3....",26.27,25.98,25.86,26.12,26.10,26.28,...,26.66,26.59,26.77,27.08,27.00,26.66,26.86,26.96,27.36,26.85
4,1177,Sokoto,NWZ,"POLYGON ((4.63265 11.67143, 4.62886 11.66810, ...",28.86,28.37,27.86,28.71,28.04,28.55,...,29.30,29.07,28.95,29.21,29.04,28.87,29.02,28.97,29.38,28.58


In [31]:
gdf.columns

Index(['uniq_id', 'statename', 'geozone', 'geometry', '1990', '1991', '1992',
       '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001',
       '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010',
       '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019',
       '2020', '2021', '2022'],
      dtype='object')

In [33]:
# Save Cleaned Data

gdf.to_file("alt_data/cleaned_avg_temp_by_ng_states.shp", driver="ESRI Shapefile")